# Analyze Annotation Progress

Reads the Firebase RTDB export and reports alignment stats per video.

In [1]:
import json
from pathlib import Path

ROOT = Path(".").resolve().parent.parent
EXPORT_PATH = ROOT / "data/firebase/latest.json"

with open(EXPORT_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

videos = data.get("videos", {})
video_captions = data.get("video_captions", {})

# Firebase may store videos as list if keys are sequential integers
if isinstance(videos, list):
    videos = {str(i): v for i, v in enumerate(videos) if v is not None}

print(f"{len(videos)} videos, {len(video_captions)} caption sets")

26 videos, 26 caption sets


## Per-Video Stats

In [2]:
total_aligned = 0
total_captions = 0
total_aligned_dur = 0.0
total_dur = 0.0

rows = []

for yt_id, cap_data in video_captions.items():
    captions = cap_data if isinstance(cap_data, list) else cap_data.get("captions", [])
    
    n_total = len(captions)
    n_aligned = 0
    dur_aligned = 0.0
    dur_total = 0.0
    
    for c in captions:
        if not isinstance(c, dict):
            continue
        d = c.get("end_time", 0) - c.get("start_time", 0)
        dur_total += d
        if c.get("aligned"):
            n_aligned += 1
            dur_aligned += d
    
    # Find assigned_to from videos list
    assigned = ""
    for v in videos.values():
        if isinstance(v, dict) and yt_id in v.get("url", ""):
            assigned = v.get("assigned_to", "")
            break
    
    pct = (n_aligned / n_total * 100) if n_total else 0
    rows.append((yt_id, assigned, n_aligned, n_total, pct, dur_aligned, dur_total))
    
    total_aligned += n_aligned
    total_captions += n_total
    total_aligned_dur += dur_aligned
    total_dur += dur_total

# Sort by aligned duration descending
rows.sort(key=lambda r: r[5], reverse=True)

print(f"{'Video ID':<15} {'Assigned':<15} {'Aligned':>8} {'Total':>6} {'%':>6}  {'Dur (min)':>10} {'Total (min)':>12}")
print("-" * 85)
for yt_id, assigned, n_al, n_tot, pct, dur_al, dur_tot in rows:
    print(f"{yt_id:<15} {assigned:<15} {n_al:>8} {n_tot:>6} {pct:>5.1f}%  {dur_al/60:>10.1f} {dur_tot/60:>12.1f}")
print("-" * 85)
pct_total = (total_aligned / total_captions * 100) if total_captions else 0
print(f"{'TOTAL':<15} {'':<15} {total_aligned:>8} {total_captions:>6} {pct_total:>5.1f}%  {total_aligned_dur/60:>10.1f} {total_dur/60:>12.1f}")
print(f"\nAligned: {total_aligned_dur/60:.1f} min ({total_aligned_dur/3600:.2f} hours)")

Video ID        Assigned         Aligned  Total      %   Dur (min)  Total (min)
-------------------------------------------------------------------------------------
d37lwXaSjs4     Volosnka             502    516  97.3%        41.4         41.8
jj5jiyl2mh0     Iriha2025            423    446  94.8%        37.1         38.0
IOflFDS2biE     Iriha2025            482    527  91.5%        36.6         38.3
0ULOz5HM4pA     Volosnka             236    251  94.0%        30.0         31.3
yPYU48eSeBg     LS04071977           292    301  97.0%        27.6         28.0
9NMtlqDBY_s     Volosnka             354    366  96.7%        27.2         27.9
VVjY5HVY0jg     Volosnka             347    366  94.8%        26.8         27.5
cNT6ajjEwVU     Iriha2025            174    187  93.0%        26.2         26.9
4FUDnWC9UJA     Volosnka             308    318  96.9%        25.4         25.9
6O0ZiSgKJNc     Volosnka             249    268  92.9%        22.3         23.5
SG9xYYOLBNI     Iriha2025         

## Temporal Analysis: Changes Across Exports

Diffs all Firebase exports chronologically to detect:
- **Text edits** — captions whose text changed between exports
- **Boundary shifts** — clips whose start/end times changed
- **New/removed clips** — clips added or dropped

This tells you when to regenerate `annotations.csv` and downstream artifacts (poses, splits).

In [3]:
import re
from collections import defaultdict

EXPORTS_DIR = ROOT / "data/firebase"
export_files = sorted(EXPORTS_DIR.glob("*.json"))
# Exclude latest.json (symlink) to avoid double-counting
export_files = [f for f in export_files if f.name != "latest.json"]

def extract_video_id(url):
    m = re.search(r'(?:v=|youtu\.be/)([a-zA-Z0-9_-]{11})', url)
    return m.group(1) if m else None

def load_export(path):
    """Load export and return {yt_id: [(idx, text, start, end, aligned), ...]}."""
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    vids = data.get("videos", {})
    if isinstance(vids, list):
        vids = {str(i): v for i, v in enumerate(vids) if v is not None}

    completed_ids = set()
    for v in vids.values():
        if isinstance(v, dict) and v.get("complete"):
            yt_id = extract_video_id(v.get("url", ""))
            if yt_id:
                completed_ids.add(yt_id)

    vc = data.get("video_captions", {})
    result = {}
    for yt_id, cap_data in vc.items():
        captions = cap_data if isinstance(cap_data, list) else cap_data.get("captions", cap_data)
        clips = []
        for idx, c in enumerate(captions):
            if not isinstance(c, dict):
                continue
            clips.append({
                "idx": idx,
                "text": c.get("text", "").strip(),
                "start": round(c.get("start_time", 0), 3),
                "end": round(c.get("end_time", 0), 3),
                "aligned": bool(c.get("aligned")),
            })
        result[yt_id] = clips
    return result, completed_ids

exports = []
for f in export_files:
    date = f.stem  # e.g. "2026-03-13"
    clips, completed = load_export(f)
    exports.append({"date": date, "path": f, "clips": clips, "completed": completed})

print(f"Loaded {len(exports)} exports: {exports[0]['date']} .. {exports[-1]['date']}")
for e in exports:
    total_clips = sum(len(v) for v in e["clips"].values())
    total_aligned = sum(sum(1 for c in v if c["aligned"]) for v in e["clips"].values())
    print(f"  {e['date']}: {len(e['clips'])} videos, {total_aligned}/{total_clips} aligned, {len(e['completed'])} complete")

Loaded 14 exports: 2025-03-11 .. 2026-04-26
  2025-03-11: 13 videos, 1525/3195 aligned, 0 complete
  2026-03-13: 13 videos, 2219/3195 aligned, 0 complete
  2026-03-15: 13 videos, 2850/3195 aligned, 0 complete
  2026-03-19: 13 videos, 2850/3195 aligned, 1 complete
  2026-03-27: 17 videos, 3927/4049 aligned, 4 complete
  2026-04-04: 26 videos, 5200/6801 aligned, 14 complete
  2026-04-05: 26 videos, 5200/6801 aligned, 14 complete
  2026-04-13: 26 videos, 5186/6801 aligned, 13 complete
  2026-04-14: 26 videos, 5186/6801 aligned, 13 complete
  2026-04-17: 26 videos, 5159/6801 aligned, 25 complete
  2026-04-18: 26 videos, 5159/6801 aligned, 25 complete
  2026-04-24: 26 videos, 5157/6801 aligned, 25 complete
  2026-04-25: 26 videos, 5160/6801 aligned, 25 complete
  2026-04-26: 26 videos, 5164/6801 aligned, 25 complete


### Changes Between Consecutive Exports

In [4]:
def diff_exports(old, new):
    """Compare two exports, return per-video change summary."""
    all_vids = set(old["clips"].keys()) | set(new["clips"].keys())
    changes = {}

    for vid in sorted(all_vids):
        old_clips = {c["idx"]: c for c in old["clips"].get(vid, [])}
        new_clips = {c["idx"]: c for c in new["clips"].get(vid, [])}

        text_edits = []
        boundary_shifts = []
        alignment_changes = []
        added = set(new_clips.keys()) - set(old_clips.keys())
        removed = set(old_clips.keys()) - set(new_clips.keys())

        for idx in set(old_clips.keys()) & set(new_clips.keys()):
            o, n = old_clips[idx], new_clips[idx]
            if o["text"] != n["text"]:
                text_edits.append(idx)
            if abs(o["start"] - n["start"]) > 0.01 or abs(o["end"] - n["end"]) > 0.01:
                boundary_shifts.append(idx)
            if o["aligned"] != n["aligned"]:
                alignment_changes.append(idx)

        if text_edits or boundary_shifts or alignment_changes or added or removed:
            changes[vid] = {
                "text_edits": text_edits,
                "boundary_shifts": boundary_shifts,
                "alignment_changes": alignment_changes,
                "added": added,
                "removed": removed,
            }
    return changes

# Diff consecutive exports
for i in range(1, len(exports)):
    old, new = exports[i - 1], exports[i]
    changes = diff_exports(old, new)

    if not changes:
        print(f"\n{old['date']} -> {new['date']}: no changes")
        continue

    total_text = sum(len(c["text_edits"]) for c in changes.values())
    total_boundary = sum(len(c["boundary_shifts"]) for c in changes.values())
    total_align = sum(len(c["alignment_changes"]) for c in changes.values())
    total_added = sum(len(c["added"]) for c in changes.values())
    total_removed = sum(len(c["removed"]) for c in changes.values())

    print(f"\n{old['date']} -> {new['date']}: "
          f"{total_text} text edits, {total_boundary} boundary shifts, "
          f"{total_align} alignment changes, +{total_added}/-{total_removed} clips")

    for vid, c in sorted(changes.items()):
        parts = []
        if c["text_edits"]:
            parts.append(f"{len(c['text_edits'])} text")
        if c["boundary_shifts"]:
            parts.append(f"{len(c['boundary_shifts'])} boundary")
        if c["alignment_changes"]:
            parts.append(f"{len(c['alignment_changes'])} align")
        if c["added"]:
            parts.append(f"+{len(c['added'])} new")
        if c["removed"]:
            parts.append(f"-{len(c['removed'])} removed")
        print(f"  {vid}: {', '.join(parts)}")


2025-03-11 -> 2026-03-13: 4 text edits, 0 boundary shifts, 694 alignment changes, +0/-0 clips
  VVjY5HVY0jg: 3 text, 326 align
  eYEK-n2alOA: 1 text, 183 align
  g5Az2FHUQBU: 185 align

2026-03-13 -> 2026-03-15: 2 text edits, 0 boundary shifts, 631 alignment changes, +0/-0 clips
  4FUDnWC9UJA: 2 text, 315 align
  VVjY5HVY0jg: 22 align
  yPYU48eSeBg: 294 align

2026-03-15 -> 2026-03-19: no changes

2026-03-19 -> 2026-03-27: 10 text edits, 1 boundary shifts, 249 alignment changes, +854/-0 clips
  0ULOz5HM4pA: +251 new
  82dy0zC6X_8: +114 new
  9NMtlqDBY_s: +366 new
  A2hCZVvtUSE: +123 new
  Nyykyn4FpNo: 10 text, 209 align
  SG9xYYOLBNI: 1 boundary, 38 align
  VVjY5HVY0jg: 2 align

2026-03-27 -> 2026-04-04: 0 text edits, 0 boundary shifts, 0 alignment changes, +2752/-0 clips
  2Nnz697BVTw: +85 new
  2QDHEz-xVq8: +1389 new
  K9ouFMtz-s8: +64 new
  KUDt_SKkPUE: +151 new
  MSqpwfErl34: +122 new
  Q3yRVXmZdGQ: +152 new
  cNT6ajjEwVU: +187 new
  jj5jiyl2mh0: +446 new
  uGMgleLkjho: +156 new



### Per-Video: Last Changed Export

Shows when each video was last modified (text, boundaries, or alignment). Videos changed after the annotations CSV was generated need a pipeline re-run.

In [5]:
import os

# Track last-changed date per video, and what kind of change
last_changed = {}  # vid -> {"date": str, "text": int, "boundary": int, "align": int}

for i in range(1, len(exports)):
    changes = diff_exports(exports[i - 1], exports[i])
    date = exports[i]["date"]
    for vid, c in changes.items():
        last_changed[vid] = {
            "date": date,
            "text": last_changed.get(vid, {}).get("text", 0) + len(c["text_edits"]),
            "boundary": last_changed.get(vid, {}).get("boundary", 0) + len(c["boundary_shifts"]),
            "align": last_changed.get(vid, {}).get("align", 0) + len(c["alignment_changes"]),
        }

# Check annotations.csv modification time
annotations_path = ROOT / "data/usl-suspilne/annotations.csv"
if annotations_path.exists():
    from datetime import datetime
    mtime = datetime.fromtimestamp(os.path.getmtime(annotations_path))
    ann_date = mtime.strftime("%Y-%m-%d")
    print(f"annotations.csv last generated: {ann_date}")
else:
    ann_date = "0000-00-00"
    print("annotations.csv not found")

print(f"\n{'Video ID':<15} {'Last Changed':>13} {'Text Edits':>11} {'Boundary':>9} {'Align':>6}  {'Stale?':>6}")
print("-" * 70)
for vid in sorted(last_changed.keys()):
    c = last_changed[vid]
    stale = "YES" if c["date"] > ann_date else ""
    print(f"{vid:<15} {c['date']:>13} {c['text']:>11} {c['boundary']:>9} {c['align']:>6}  {stale:>6}")

# Videos never changed (stable since first export)
all_vids = set()
for e in exports:
    all_vids.update(e["clips"].keys())
never_changed = all_vids - set(last_changed.keys())
if never_changed:
    print(f"\nNever changed ({len(never_changed)} videos): {', '.join(sorted(never_changed))}")

stale_vids = [v for v, c in last_changed.items() if c["date"] > ann_date]
if stale_vids:
    print(f"\n!! {len(stale_vids)} video(s) changed after annotations.csv — re-run build_annotations_from_firebase.py")
else:
    print(f"\nAll videos up to date with annotations.csv")

annotations.csv not found

Video ID         Last Changed  Text Edits  Boundary  Align  Stale?
----------------------------------------------------------------------
0ULOz5HM4pA        2026-04-18          15         0      0     YES
2Nnz697BVTw        2026-04-25          70        85      4     YES
2QDHEz-xVq8        2026-04-04           0         0      0     YES
4FUDnWC9UJA        2026-04-18          46         0    322     YES
6O0ZiSgKJNc        2026-04-18          14         0      0     YES
82dy0zC6X_8        2026-04-18          20         0      4     YES
9NMtlqDBY_s        2026-04-18          29         0      1     YES
A2hCZVvtUSE        2026-04-18          13         0      1     YES
IOflFDS2biE        2026-04-25         273       480     46     YES
K9ouFMtz-s8        2026-04-26          57        62      8     YES
KUDt_SKkPUE        2026-04-18          31         0      0     YES
MSqpwfErl34        2026-04-26          44        18      2     YES
Nyykyn4FpNo        2026-04-25  